# Neural approximate-BR calibration

This notebook tests whether neural best-response training produces a predictable lower-bound curve. It uses a small game where both the optimal response ceiling and every learned responder checkpoint can be evaluated exactly.

The central quantity is the oracle gap

$$G(t)=E_{\mathrm{exact}}-L_{\mathrm{best}}(t),$$

where $L_{\mathrm{best}}(t)$ combines the best response found so far in each role. A useful approximate-BR method should make this gap fall smoothly and predictably.

In [ ]:
from pathlib import Path
import gc
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.optimize import curve_fit

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.serialization import load_policy
from liars_poker.policies.tabular_dense import DenseTabularPolicy, mix_dense
from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.algo.br_neural import NeuralBRTrainer
from liars_poker.eval.match import exact_eval_tabular

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__)
print('device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## Target ladder

We mix a saved strong FSP policy with the uniform dense policy. `mix_dense` performs the reach-correct mixture rather than independently averaging action probabilities. We calculate each candidate's actual exploitability before choosing weak, intermediate and strong targets.

In [ ]:
spec = GameSpec(
    ranks=3,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair'),
    suit_symmetry=True,
)

strong_dir = (
    REPO_ROOT / 'artifacts' / 'benchmark_runs' / 'fsp_runs'
    / 'r3_s4_h2_hp_ss___20260102-022533' / 'policy'
)
strong_policy, strong_spec = load_policy(strong_dir)
assert strong_spec == spec
assert isinstance(strong_policy, DenseTabularPolicy)

uniform_policy = DenseTabularPolicy(spec)
candidate_weights = (0.0, 0.25, 0.5, 0.75, 1.0)
candidate_targets = {
    weight: mix_dense(strong_policy, uniform_policy, weight)
    for weight in candidate_weights
}
print('claims:', len(strong_policy.rules.claims))
print('hands:', len(strong_policy.hands))

## Exact learned-responder evaluation

The earlier approximate-BR notebook calculated the exact **optimal** response ceiling but evaluated learned responders by Monte Carlo. Here each learned Q policy is compiled to a deterministic dense policy and evaluated exactly in both seats, removing evaluation noise from the calibration curves.

In [ ]:
def compile_q_to_dense(policy, batch_size=65_536):
    dense = DenseTabularPolicy(policy.spec)
    dense.S.fill(0.0)
    n_hands = len(dense.hands)
    histories_per_batch = max(1, int(batch_size) // n_hands)
    rank_dim = policy.spec.ranks
    input_dim = policy.encoder.input_dim
    action_dim = policy.encoder.action_dim
    claim_bits = np.arange(policy.encoder.k, dtype=np.int64)
    hand_features = policy.encoder.encode_hands(dense.hands, ())[:, :rank_dim]

    with torch.inference_mode():
        for pid in (0, 1):
            actor_hids = np.flatnonzero((dense.popcount & 1) == pid)
            model = policy._model(pid)
            for start in range(0, len(actor_hids), histories_per_batch):
                hids = actor_hids[start:start + histories_per_batch]
                history_bits = (
                    (hids[:, None].astype(np.int64) >> claim_bits[None, :]) & 1
                ).astype(np.float32)
                features = np.empty(
                    (len(hids), n_hands, input_dim), dtype=np.float32
                )
                features[:, :, :rank_dim] = hand_features[None, :, :]
                features[:, :, rank_dim:] = history_bits[:, None, :]

                x = torch.from_numpy(features.reshape(-1, input_dim)).to(policy.device)
                q = model(x).reshape(len(hids), n_hands, action_dim)
                legal = torch.from_numpy(dense.legal_mask[hids]).to(policy.device)
                best_cols = q.masked_fill(~legal[:, None, :], -torch.inf).argmax(dim=2)
                best_cols = best_cols.cpu().numpy()

                block = np.zeros(
                    (len(hids), n_hands, action_dim), dtype=np.float32
                )
                h_idx = np.arange(len(hids))[:, None]
                hand_idx = np.arange(n_hands)[None, :]
                block[h_idx, hand_idx, best_cols] = 1.0
                dense.S[hids] = block

    dense.recompute_likelihoods()
    return dense


def exact_oracle(target):
    _, meta = best_response_dense(
        spec, target, debug=False, store_state_values=False
    )
    p_first, p_second = meta['computer'].exploitability()
    return {
        'exact_first': float(p_first),
        'exact_second': float(p_second),
        'exact_exploitability': float(p_first + p_second - 1.0),
    }


def exact_learned_response(trainer, target):
    responder = compile_q_to_dense(trainer.policy())
    as_first = exact_eval_tabular(spec, responder, target)
    as_second = exact_eval_tabular(spec, target, responder)
    p_first = float(as_first['P1'])
    p_second = float(as_second['P2'])
    return {
        'p_first': p_first,
        'p_second': p_second,
        'discovered_exploitability': p_first + p_second - 1.0,
    }

In [ ]:
ladder_rows = []
target_oracles = {}
for weight, target in candidate_targets.items():
    oracle = exact_oracle(target)
    target_oracles[weight] = oracle
    ladder_rows.append({'strong policy weight': weight, **oracle})

target_ladder = pd.DataFrame(ladder_rows).sort_values(
    'exact_exploitability', ascending=False
).reset_index(drop=True)
selected_rows = {
    'weak': target_ladder.iloc[0],
    'intermediate': target_ladder.iloc[len(target_ladder) // 2],
    'strong': target_ladder.iloc[-1],
}
target_suite = {
    label: {
        'weight': float(row['strong policy weight']),
        'policy': candidate_targets[float(row['strong policy weight'])],
        'oracle': target_oracles[float(row['strong policy weight'])],
    }
    for label, row in selected_rows.items()
}

display(target_ladder)
pd.DataFrame({
    label: {'strong policy weight': item['weight'], **item['oracle']}
    for label, item in target_suite.items()
}).T

## Calibration runner

Training time excludes exact evaluations. At every geometric budget we record the current response and the cumulative best response found separately in each role. Combining the best role values gives a monotone discovered lower bound.

In [ ]:
BASE_TRAINER_KWARGS = dict(
    hidden_sizes=(256, 256),
    device=device,
    replay_capacity=250_000,
    batch_size=1024,
    learning_rate=1e-3,
    warmup_transitions=5_000,
    target_update_every=500,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_decisions=200_000,
)
EPISODES_PER_ROLE = 2048
ROLLOUT_BATCH_SIZE = 2048


def run_exact_calibration(target, oracle, config, seed, budgets_s):
    trainer = NeuralBRTrainer(
        spec, target, seed=seed, **BASE_TRAINER_KWARGS, **config
    )
    measured_s = 0.0
    best_first = -np.inf
    best_second = -np.inf
    rows = []
    last_train = None

    def capture(requested_budget_s):
        nonlocal best_first, best_second
        learned = exact_learned_response(trainer, target)
        best_first = max(best_first, learned['p_first'])
        best_second = max(best_second, learned['p_second'])
        best_lower = best_first + best_second - 1.0
        gap = max(0.0, oracle['exact_exploitability'] - best_lower)
        row = {
            'seed': seed,
            'requested_budget_s': float(requested_budget_s),
            'measured_training_s': measured_s,
            'iteration': trainer.iteration,
            **learned,
            'best_p_first': best_first,
            'best_p_second': best_second,
            'best_discovered_exploitability': best_lower,
            'oracle_gap': gap,
            'relative_oracle_gap': gap / max(oracle['exact_exploitability'], 1e-12),
        }
        if last_train is not None:
            for role in (0, 1):
                role_record = last_train['roles'][role]
                row[f'role_{role + 1}_loss'] = role_record['loss']
                row[f'role_{role + 1}_replay_seen'] = role_record['replay_seen']
                row[f'role_{role + 1}_epsilon'] = role_record['epsilon']
        rows.append(row)
        print(
            f"seed={seed:2d} train={measured_s:7.2f}s iter={trainer.iteration:4d} "
            f"lower={best_lower:.6f} gap={gap:.6f}"
        )

    budgets = list(map(float, budgets_s))
    next_budget = 0
    while next_budget < len(budgets):
        if measured_s >= budgets[next_budget]:
            capture(budgets[next_budget])
            next_budget += 1
            continue
        start = time.perf_counter()
        last_train = trainer.run_iteration(
            episodes_per_role=EPISODES_PER_ROLE,
            rollout_batch_size=ROLLOUT_BATCH_SIZE,
        )
        measured_s += time.perf_counter() - start

    result = pd.DataFrame(rows)
    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def normalized_auc(frame, column):
    ordered = frame.sort_values('measured_training_s')
    x = ordered['measured_training_s'].to_numpy()
    y = ordered[column].to_numpy()
    if len(x) < 2 or x[-1] <= x[0]:
        return float(y[-1])
    return float(np.trapezoid(y, x) / (x[-1] - x[0]))


def known_exact_power_alpha(frame):
    usable = frame[
        (frame['measured_training_s'] > 0) & (frame['oracle_gap'] > 1e-10)
    ]
    if len(usable) < 3:
        return np.nan
    slope, _ = np.polyfit(
        np.log(usable['measured_training_s']), np.log(usable['oracle_gap']), 1
    )
    return float(-slope)

## Stage A: choose the responder training method

The intermediate target is used to compare sampled actions, one-step expansion of all responder actions, and greater fitting effort for expanded data. All variants receive the same measured training-time budget.

In [ ]:
STAGE_A_BUDGETS_S = (0, 1, 2, 4, 8, 15, 30, 60, 90)
STAGE_A_SEEDS = (7, 17, 27)
STAGE_A_CONFIGS = {
    'sampled; 100 steps': {'expansion': 'sampled', 'train_steps': 100},
    'all actions; 100 steps': {'expansion': 'all_actions', 'train_steps': 100},
    'all actions; 300 steps': {'expansion': 'all_actions', 'train_steps': 300},
}

intermediate = target_suite['intermediate']
stage_a_frames = []
for variant, config in STAGE_A_CONFIGS.items():
    for seed in STAGE_A_SEEDS:
        print(f'\n=== {variant}; seed={seed} ===')
        frame = run_exact_calibration(
            intermediate['policy'], intermediate['oracle'], config,
            seed, STAGE_A_BUDGETS_S,
        )
        frame['variant'] = variant
        stage_a_frames.append(frame)

stage_a = pd.concat(stage_a_frames, ignore_index=True)

In [ ]:
stage_a_rows = []
for (variant, seed), frame in stage_a.groupby(['variant', 'seed']):
    final = frame.sort_values('measured_training_s').iloc[-1]
    stage_a_rows.append({
        'variant': variant,
        'seed': seed,
        'final discovered lower bound': final['best_discovered_exploitability'],
        'final oracle gap': final['oracle_gap'],
        'oracle-gap normalized AUC': normalized_auc(frame, 'relative_oracle_gap'),
        'known-exact power alpha': known_exact_power_alpha(frame),
    })

stage_a_summary = pd.DataFrame(stage_a_rows)
display(stage_a_summary.set_index(['variant', 'seed']))
display(stage_a_summary.groupby('variant').agg(['mean', 'std']))

winner_name = (
    stage_a_summary.groupby('variant')['oracle-gap normalized AUC'].mean().idxmin()
)
winner_config = STAGE_A_CONFIGS[winner_name]
print('Stage A winner:', winner_name, winner_config)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for variant, group in stage_a.groupby('variant'):
    curve = group.groupby('requested_budget_s').agg(
        measured_training_s=('measured_training_s', 'mean'),
        lower=('best_discovered_exploitability', 'mean'),
        gap=('oracle_gap', 'mean'),
    )
    positive = curve['measured_training_s'] > 0
    axes[0].plot(
        curve.loc[positive, 'measured_training_s'], curve.loc[positive, 'lower'],
        marker='o', label=variant,
    )
    axes[1].loglog(
        curve.loc[positive, 'measured_training_s'],
        curve.loc[positive, 'gap'].clip(lower=1e-10),
        marker='o', label=variant,
    )

axes[0].axhline(
    intermediate['oracle']['exact_exploitability'],
    color='black', linestyle='--', label='exact ceiling',
)
axes[0].set_xscale('log')
axes[0].set_title('Best discovered exploitability')
axes[1].set_title('Exact oracle gap')
for ax in axes:
    ax.set_xlabel('Measured training seconds')
    ax.grid(alpha=0.3, which='both')
    ax.legend()
plt.tight_layout()

## Stage B: test the winning method across target strength

The same responder setup is trained against weak, intermediate and strong targets. We want the cumulative best lower bound to rise, the exact gap to decline approximately smoothly on log-log axes, and early curves to contain information about the eventual asymptote.

In [ ]:
STAGE_B_BUDGETS_S = (0, 1, 2, 4, 8, 15, 30, 60, 120, 240)
STAGE_B_SEEDS = (7, 17, 27)

stage_b_frames = []
for target_name, target in target_suite.items():
    for seed in STAGE_B_SEEDS:
        print(f'\n=== target={target_name}; seed={seed}; {winner_name} ===')
        frame = run_exact_calibration(
            target['policy'], target['oracle'], winner_config,
            seed, STAGE_B_BUDGETS_S,
        )
        frame['target'] = target_name
        frame['exact_exploitability'] = target['oracle']['exact_exploitability']
        stage_b_frames.append(frame)

stage_b = pd.concat(stage_b_frames, ignore_index=True)

In [ ]:
stage_b_rows = []
for (target_name, seed), frame in stage_b.groupby(['target', 'seed']):
    final = frame.sort_values('measured_training_s').iloc[-1]
    stage_b_rows.append({
        'target': target_name,
        'seed': seed,
        'exact exploitability': final['exact_exploitability'],
        'final discovered lower bound': final['best_discovered_exploitability'],
        'final oracle gap': final['oracle_gap'],
        'fraction recovered': (
            final['best_discovered_exploitability']
            / max(final['exact_exploitability'], 1e-12)
        ),
        'oracle-gap normalized AUC': normalized_auc(frame, 'relative_oracle_gap'),
        'known-exact power alpha': known_exact_power_alpha(frame),
    })

stage_b_summary = pd.DataFrame(stage_b_rows)
display(stage_b_summary.set_index(['target', 'seed']))
display(stage_b_summary.groupby('target').agg(['mean', 'std']))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for target_name, group in stage_b.groupby('target'):
    curve = group.groupby('requested_budget_s').agg(
        measured_training_s=('measured_training_s', 'mean'),
        current=('discovered_exploitability', 'mean'),
        best=('best_discovered_exploitability', 'mean'),
        gap=('oracle_gap', 'mean'),
        exact=('exact_exploitability', 'mean'),
    )
    positive = curve['measured_training_s'] > 0
    x = curve.loc[positive, 'measured_training_s']
    axes[0].plot(x, curve.loc[positive, 'current'], marker='o', label=target_name)
    axes[1].plot(x, curve.loc[positive, 'best'], marker='o', label=target_name)
    axes[1].axhline(curve['exact'].iloc[-1], linestyle='--', alpha=0.45)
    axes[2].loglog(
        x, curve.loc[positive, 'gap'].clip(lower=1e-10),
        marker='o', label=target_name,
    )

axes[0].set_title('Raw current discovered exploitability')
axes[1].set_title('Monotone best discovered lower bound')
axes[2].set_title('Gap to exact optimal response')
for ax in axes[:2]:
    ax.set_xscale('log')
for ax in axes:
    ax.set_xlabel('Measured training seconds')
    ax.grid(alpha=0.3, which='both')
    ax.legend()
plt.tight_layout()

## Can an early curve predict the true exploitability?

This final diagnostic pretends that the exact ceiling is unknown. It fits

$$L(t)=E_\infty-c(t+1)^{-\alpha}$$

using only the first 60% of checkpoints, then compares the inferred asymptote with the known exact exploitability. This is a heuristic extrapolation test, not a certified bound.

In [ ]:
def asymptotic_curve(t, asymptote, scale, alpha):
    return asymptote - scale * np.power(t + 1.0, -alpha)


def fit_early_asymptote(frame, fraction=0.6):
    ordered = frame.sort_values('measured_training_s')
    ordered = ordered[ordered['measured_training_s'] > 0]
    n_fit = max(4, int(np.ceil(len(ordered) * fraction)))
    fit = ordered.iloc[:n_fit]
    t = fit['measured_training_s'].to_numpy(dtype=float)
    y = fit['best_discovered_exploitability'].to_numpy(dtype=float)
    lower_asymptote = float(y.max())
    initial_asymptote = min(0.999, lower_asymptote + max(0.01, np.ptp(y)))
    initial_scale = max(1e-4, initial_asymptote - y[0])
    params, _ = curve_fit(
        asymptotic_curve, t, y,
        p0=(initial_asymptote, initial_scale, 0.5),
        bounds=((lower_asymptote, 0.0, 0.01), (1.0, 10.0, 3.0)),
        maxfev=20_000,
    )
    return {
        'fitted asymptote': float(params[0]),
        'fitted scale': float(params[1]),
        'fitted alpha': float(params[2]),
        'fit through requested budget s': float(fit['requested_budget_s'].iloc[-1]),
    }


extrapolation_rows = []
for (target_name, seed), frame in stage_b.groupby(['target', 'seed']):
    exact = float(frame['exact_exploitability'].iloc[0])
    try:
        fitted = fit_early_asymptote(frame)
        extrapolation_rows.append({
            'target': target_name,
            'seed': seed,
            'exact exploitability': exact,
            **fitted,
            'asymptote error': fitted['fitted asymptote'] - exact,
        })
    except (RuntimeError, ValueError) as exc:
        extrapolation_rows.append({
            'target': target_name,
            'seed': seed,
            'exact exploitability': exact,
            'fit error': str(exc),
        })

extrapolation = pd.DataFrame(extrapolation_rows)
display(extrapolation.set_index(['target', 'seed']))

valid = extrapolation.dropna(subset=['fitted asymptote'])
fig, ax = plt.subplots(figsize=(6, 5))
for target_name, group in valid.groupby('target'):
    ax.scatter(
        group['exact exploitability'], group['fitted asymptote'],
        s=60, label=target_name,
    )
limit = max(
    0.01,
    valid[['exact exploitability', 'fitted asymptote']].to_numpy().max(),
)
ax.plot([0, limit], [0, limit], color='black', linestyle='--', label='perfect prediction')
ax.set_xlabel('Known exact exploitability')
ax.set_ylabel('Early-curve fitted asymptote')
ax.set_title('Does early approximate-BR progress extrapolate?')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()